In [ ]:
# Instalaivimas reikalingų paketų
#!pip install -U "typing_extensions>=4.7.0"
#!pip install -U "lmdeploy>=0.9.1"
#!pip install lmdeploy>=0.9.1
#!pip install -U transformers accelerate torch torchvision pillow

print("Instaliavimas baigtas")

In [ ]:
# Perspėjimų ignoravimas
import warnings
warnings.filterwarnings("ignore")  # Ignore all warnings
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText
from lmdeploy import pipeline, PytorchEngineConfig
from lmdeploy.vl import load_image
print("Importavimas baigtas")

In [ ]:
# Modelių naudojimas
model = 'OpenGVLab/InternVL3_5-38B'
#OpenGVLab/InternVL3_5-8B
#OpenGVLab/InternVL3_5-14B
#OpenGVLab/InternVL3_5-30B-A3B
#OpenGVLab/InternVL3_5-38B
pipe = pipeline(model, backend_config=PytorchEngineConfig(session_len=4096, tp=4))
print("Modelio krovimas baigtas")

In [ ]:
import pandas as pd
import os
import re
from PIL import Image
df = pd.read_csv('flicker8k_edited.csv', sep=';')
import numpy as np

# Išsivalome lentelę
columns_to_clear = [
    'BLEU', 'ROGUE', 'METEOR',
    'BERTscore Precision', 'BERTscore Recall', 'BERTscore F1', 'SentenceBERT'
]
for col in columns_to_clear:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: '')

image_folder = "Flickr8k"

#for index, row in df.head(20).iterrows():
for index, row in df.iterrows():
    filename = row["Image Name"]
    image_path = os.path.join(image_folder, filename)
    try:
        image = Image.open(image_path)
        generated_caption = pipe(('Keep it short. No more than 10 words. The text is the caption for the image. Do not write any additional text.', image))
        #print(generated_caption.text)
        
        # Įrašymas į failą
        df.at[index, "Generated Caption"] = generated_caption.text

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        df.at[index, "Generated Caption"] = f"Error: {e}"

# Saugojimas
df.to_csv("Results/flicker8k_en_24.csv", sep=';', index=False)
print("Generavimas baigtas")